In [ ]:
%%writefile ambiente.h
// Contiene: i parametri, gli ostacoli (ostacoli e fori), la maschera della griglia,
// le sorgenti della passata 1 (bordo degli ostacoli)
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>
#include <cuda_runtime.h>   // calcolo lentezza su GPU

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Zona { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1


// AMBIENTE: per cambiarlo si modifica solo questa parte
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

const Punto A = { -0.8,  0.0, -0.6 };                    // partenza del drone: primo scomparto, in basso
const Punto B = {  0.85, 0.45,  0.3 };                   // arrivo (sorgente del FIM): terzo scomparto, lontano dai muri

// Cubo 3D: [-1, 1]^3
const double SP = 0.04;                                  // spessore delle pareti della stanza
const std::vector<Zona> PARETI = {                      // le 6 pareti della stanza (pavimento e soffitto compresi)
    { -1.0, -1.0 + SP, -1.0, 1.0, -1.0, 1.0 }, { 1.0 - SP, 1.0, -1.0, 1.0, -1.0, 1.0 },         // lungo x
    { -1.0, 1.0, -1.0, -1.0 + SP, -1.0, 1.0 }, { -1.0, 1.0, 1.0 - SP, 1.0, -1.0, 1.0 },         // lungo y
    { -1.0, 1.0, -1.0, 1.0, -1.0, -1.0 + SP }, { -1.0, 1.0, -1.0, 1.0, 1.0 - SP, 1.0 },         // lungo z
};
// Tre scomparti separati da due muri; ogni muro ha una porta centrale (che si chiude durante il volo) e una finestra
const Zona MURO_1 = { -0.45, -0.35, -1.00, 1.00, -1.00, 1.00 };          // primo muro (x = -0.4)
const Zona MURO_2 = { -0.05, 0.05, -1.00, 1.00, -1.00, 1.00 };          // secondo muro (x = 0)
const Zona PORTA_1    = { -0.45, -0.35, -0.25, 0.25, -1.00, 0.10 };   // porta centrale del primo muro (dal pavimento)
const Zona FINESTRA_1 = { -0.45, -0.35, 0.45, 0.75, 0.30, 0.70 };   // finestra del primo muro: in alto, verso y > 0
const Zona PORTA_2    = { -0.05, 0.05, -0.25, 0.25, -1.00, 0.10 };   // porta centrale del secondo muro
const Zona FINESTRA_2 = { -0.05, 0.05, -0.75, -0.45, -0.55, -0.10 }; // finestra del secondo muro: accanto alla porta 2, alla stessa altezza, verso y < 0

// Ripianificazione: la mappa cambia durante il volo. imposta_mappa(m) sceglie la mappa attuale:
// 0 = tutte le porte aperte, 1 = porta 1 chiusa, 2 = porte 1 e 2 chiuse (restano le due finestre)
const int N_MAPPE = 3;
inline std::vector<Zona> OSTACOLI;
inline std::vector<Zona> FORI;                        // zone in cui l'ostacolo viene tolto (porte e finestre aperte)

// Mappa m e cosa cambia rispetto alla mappa m - 1. Sequenza: 0 = tutto aperto, 1 = porta 1 chiusa,
// 2 = porte 1 e 2 chiuse (le finestre restano sempre aperte)
inline void imposta_mappa(int m, std::vector<Zona>& aggiunti, std::vector<Zona>& tolti)
{
    OSTACOLI = PARETI;
    OSTACOLI.push_back(MURO_1);
    OSTACOLI.push_back(MURO_2);
    FORI = { FINESTRA_1, FINESTRA_2 };
    aggiunti.clear(); tolti.clear();
    // porta 1: aperta nella mappa 0, si chiude nella mappa 1
    if (m < 1)  FORI.push_back(PORTA_1);
    if (m == 1) aggiunti.push_back(PORTA_1);
    // porta 2: aperta fino alla mappa 1, si chiude nella mappa 2
    if (m < 2)  FORI.push_back(PORTA_2);
    if (m == 2) aggiunti.push_back(PORTA_2);
}
// Quando serve solo la mappa (per esempio per la maschera)
inline void imposta_mappa(int m) { std::vector<Zona> a, t; imposta_mappa(m, a, t); }
inline const bool MAPPA_INIZIALE = (imposta_mappa(0), true);   // all'avvio: mappa 0

// Vero se il punto (x, y, z) e' dentro la ostacolo os (bordi compresi)
inline bool dentro(const Zona& os, double x, double y, double z)
{
    return os.x0 <= x && x <= os.x1 && os.y0 <= y && y <= os.y1 && os.z0 <= z && z <= os.z1;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& os : OSTACOLI) if (dentro(os, x, y, z)) return true;
    return false;
}

// Maschera della griglia: libero = 1, ostacolo = 0 (resta INF)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    const int n = N * N * N;
    std::vector<unsigned char> mask(n);
    for (int q = 0; q < n; ++q) {                     // un nodo alla volta
    const int j = q % N, i = (q / N) % N, k = q / (N * N);
    mask[q] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;
    }
return mask;
}
// Sorgenti della passata 1: i nodi liberi che toccano un ostacolo
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                if (!mask[q]) continue;                                  // solo nodi liberi...
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // ...che hanno un vicino ostacolo
            }
    return bordo;
}

// FM2: dalla distanza d dagli ostacoli alla lentezza F = 1/v della passata 2
// v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1): F = 10 sul bordo, 1 lontano
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: 0 sul bordo, 1 oltre D_SICUREZZA (o se d = INF)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}

// La stessa lentezza del FM2 calcolata sulla GPU: un thread per nodo, d resta nella memoria della GPU.
__global__ void lentezza_fm2_gpu(const float* d, float* F, int n, float v_min, float d_sic)
{
    const int q = blockIdx.x * blockDim.x + threadIdx.x;      // il nodo di questo thread
    if (q >= n) return;                                       // thread in eccesso
    const float r = fminf(d[q] / d_sic, 1.0f);                // distanza normalizzata (tutto in float: sulla T4 il double e' circa 32 volte piu' lento)
    F[q] = 1.0f / (v_min + (1.0f - v_min) * r);
}

// Controllo iniziale: A e B non devono stare dentro un ostacolo
inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile godunov3d.h
#pragma once           // il file viene incluso una sola volta
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#include <cmath>       // fminf, sqrtf

const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
// Risolutore locale 3D (Godunov, primo ordine), comune ai tre metodi.
// __host__ __device__: compilata per la CPU (la usa il FMM) e per la GPU (Block FIM e FIM-Jacobi)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include "godunov3d.h"  // risolutore locale

typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs){heap.push(Entry(u[s], s)); state[s] = ATTIVO;}

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
%%writefile block_fim3d.cuh
#pragma once           // il file viene incluso una sola volta
// Block FIM 3D (Alg. 7) sulla GPU: blocchi 8^3 in shared memory, uscita anticipata, maschera degli ostacoli
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>       // std::min
#include <cuda_runtime.h>
#include "godunov3d.h"  // risolutore locale

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo
// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Aggiorna i blocchi di una lista (Alg. 7): lista dei blocchi attivi (Step 1, niter iterazioni) 
// oppure dei candidati (Step 2, 1 iterazione)
__global__ void update_blocks(const int* list, const int* quantity, int niter, float* u, const float* F,
                             int N, int nbx, float h, const unsigned char* mask, float eps, unsigned char* Cb_out, int* is_candidate)
{
    // quantity = 0/1; *quantity = numero di blocchi attivi
    if (quantity && blockIdx.x >= *quantity) return;
    // (es. N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (es. N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
    const int block = list[blockIdx.x];

    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o maschera
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q = fixed ? p : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);  
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
        if (!__syncthreads_or(!conv)) break;   // tutti hanno scritto; se nessun nodo e' cambiato, esci
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cb del blocco: vero solo se TUTTI i 512 nodi sono convergenti. __syncthreads_and fa l'AND di conv su tutti i thread del
    // blocco in una sola istruzione (al posto della riduzione ad albero della tesi, Fig. 3.7)
    const bool blocco_conv = __syncthreads_and(conv);
    if (tid == 0) Cb_out[block] = blocco_conv;
    // solo candidati: pronto per il giro dopo
    if (is_candidate && tid == 0) is_candidate[block] = 0;  
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (monodimensionale CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(float* d_u, const float* d_F, const unsigned char* d_mask, int N, float h,
              const std::vector<int>& srcs, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    for (int s : srcs){
    const int sk = s / (N * N), si = (s / N) % N, sj = s % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;
    }
    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    unsigned char *d_Cb, *d_Cb_step1, *d_in_list; int *d_active_list, *d_n_active;
    int *d_is_candidate, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks)); CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks)); CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int))); CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int)));
    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int))); CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks)); CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop)); CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, nullptr, niter, d_u, d_F, N, nbx, h, d_mask, eps, d_Cb_step1, nullptr);
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, 1,
                                                                                    d_u, d_F, N, nbx, h, d_mask, eps, d_Cb, d_is_candidate);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaFree(d_Cb)); CUDA_CHECK(cudaFree(d_Cb_step1));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}

In [ ]:
%%writefile jacobi_fim.cuh
// FIM-Jacobi senza lista, in CUDA 3D: a ogni iterazione tutti i nodi si aggiornano con Godunov usando i valori
// dell'iterazione prima (Jacobi), nuovo = min(vecchio, Godunov); due buffer che si alternano
#pragma once
#include <vector>
#include <utility>          // std::swap
#include <cuda_runtime.h>
#include "godunov3d.h"  // risolutore locale

const int CONTROLLO = 25;   // ogni quante iterazioni si controlla la convergenza

// Un thread per nodo: nuovo valore del nodo x dai 6 vicini dell'iterazione precedente
__global__ void jacobi_update(const float* cur, const float* F, const unsigned char* mask, float* nxt,
                              int N, float h, float eps, int* cambiato)
{
    const int x = blockIdx.x * blockDim.x + threadIdx.x;
    if (x >= N * N * N) return;                                   // thread in eccesso
    const float p = cur[x];
    if (p == 0.0f || !mask[x]) { nxt[x] = p; return; }            // sorgente o ostacolo: non cambia (gli ostacoli restano a INF)
    const int j = x % N, i = (x / N) % N, k = x / (N * N);
    const float a = fminf(j > 0 ? cur[x - 1] : INF, j < N - 1 ? cur[x + 1] : INF);              // vicini lungo x
    const float b = fminf(i > 0 ? cur[x - N] : INF, i < N - 1 ? cur[x + N] : INF);              // vicini lungo y
    const float c = fminf(k > 0 ? cur[x - N * N] : INF, k < N - 1 ? cur[x + N * N] : INF);      // vicini lungo z
    const float q = fminf(p, solve_quadratic3(a, b, c, F[x], h));                               // min(vecchio, Godunov)
    nxt[x] = q;
    if (cambiato && p - q >= eps) *cambiato = 1;                  // solo nelle iterazioni di controllo: qualcuno cambia ancora
}

// Stessa interfaccia del FIM a nodi: le sorgenti sono gia' a 0 in u
int jacobi_fim(float* d_u, const float* d_F, const unsigned char* d_mask, int N, float h, float eps, float* ms)
{
    const int n = N * N * N;
    const size_t bytes = (size_t)n * sizeof(float);
    float* d_tmp; int* d_cambiato;                                  // d_tmp: il secondo buffer, interno
    CUDA_CHECK(cudaMalloc(&d_tmp, bytes)); CUDA_CHECK(cudaMalloc(&d_cambiato, sizeof(int)));
    float *d_cur = d_u, *d_nxt = d_tmp;                              // si parte dai valori in d_u
    const int threads = 256, blocchi = (n + threads - 1) / threads;   // come il suo threads_per_block = 256
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start)); CUDA_CHECK(cudaEventCreate(&stop)); CUDA_CHECK(cudaEventRecord(start));
    int it = 0;
    while (true) {
        const bool controlla = (it % CONTROLLO == 0);
        if (controlla) CUDA_CHECK(cudaMemset(d_cambiato, 0, sizeof(int)));
        jacobi_update<<<blocchi, threads>>>(d_cur, d_F, d_mask, d_nxt, N, h, eps, controlla ? d_cambiato : nullptr);
        CUDA_CHECK(cudaGetLastError());
        std::swap(d_cur, d_nxt);                                  // il nuovo diventa il vecchio
        ++it;
        if (controlla) {                                          // sincronizzazione GPU -> CPU solo ogni CONTROLLO iterazioni
            int cambiato = 0;
            CUDA_CHECK(cudaMemcpy(&cambiato, d_cambiato, sizeof(int), cudaMemcpyDeviceToHost));
            if (!cambiato) break;
        }
    }
    CUDA_CHECK(cudaEventRecord(stop)); CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));
    CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
    if (d_cur != d_u)                                                // il risultato e' rimasto nel buffer interno:
        CUDA_CHECK(cudaMemcpy(d_u, d_cur, bytes, cudaMemcpyDeviceToDevice));   // copia GPU -> GPU, molto veloce
    CUDA_CHECK(cudaFree(d_tmp)); CUDA_CHECK(cudaFree(d_cambiato));

    return it;
}

In [ ]:
%%writefile percorso.h
// Percorso del drone e volo con i cambi di mappa (prima era nella cella Python): ricostruzione del percorso con RK4 lungo -grad u,
// istanti di passaggio, frenata davanti alla porta chiusa e file dei voli per la grafica.
// Coordinate nel cubo [-1, 1]^3; il nodo (j, i, k) e' il punto (x, y, z) = (-1 + j h, -1 + i h, -1 + k h), indice q = (k * N + i) * N + j
#pragma once
#include <cstdio>          // printf
#include <cstdlib>         // exit
#include <vector>
#include <string>          // nome del file dei voli
#include <utility>         // std::pair
#include <cmath>
#include <algorithm>       // std::clamp
#include "ambiente.h"      // Punto, B
#include "godunov3d.h"     // INF

// Operazioni sui punti (vettori 3D), per scrivere RK4 come sulla carta
inline Punto operator+(Punto a, Punto b) { return { a.x + b.x, a.y + b.y, a.z + b.z }; }
inline Punto operator-(Punto a, Punto b) { return { a.x - b.x, a.y - b.y, a.z - b.z }; }
inline Punto operator*(double s, Punto a) { return { s * a.x, s * a.y, s * a.z }; }
inline double norma(Punto a) { return std::sqrt(a.x * a.x + a.y * a.y + a.z * a.z); }

// Una mappa gia' risolta con il FM2: tutto quello che serve al drone per volare
struct Mappa {
    int N;                               // nodi per lato
    double h;                            // passo della griglia
    std::vector<float> u;                // tempo di arrivo da B (INF negli ostacoli)
    std::vector<float> F;                // lentezza del FM2
    std::vector<unsigned char> mask;     // 1 = libero, 0 = ostacolo 
    std::vector<Zona> aggiunti, tolti;   // cosa e' cambiato rispetto alla mappa precedente (vuoti per la mappa 0)
};

// Valore di f nel punto p, interpolato (trilineare) dagli 8 nodi della cella che contiene p
template <class T>
inline double interpola(const Mappa& mp, const std::vector<T>& f, Punto p)
{
    const double qx = (p.x + 1.0) / mp.h, qy = (p.y + 1.0) / mp.h, qz = (p.z + 1.0) / mp.h;   // posizione in indici, non intera
    const int j0 = std::clamp((int)std::floor(qx), 0, mp.N - 2);    // nodo in basso a sinistra della cella
    const int i0 = std::clamp((int)std::floor(qy), 0, mp.N - 2);
    const int k0 = std::clamp((int)std::floor(qz), 0, mp.N - 2);
    const double wx = qx - j0, wy = qy - i0, wz = qz - k0;          // dove sta p dentro la cella (tra 0 e 1)
    double s = 0.0;
    for (int dk = 0; dk <= 1; ++dk)
        for (int di = 0; di <= 1; ++di)
            for (int dj = 0; dj <= 1; ++dj) {
                const double peso = (dj ? wx : 1 - wx) * (di ? wy : 1 - wy) * (dk ? wz : 1 - wz);
                s += peso * f[((k0 + dk) * mp.N + i0 + di) * mp.N + j0 + dj];
            }
    return s;
}

// Vero se p e' fuori dagli ostacoli : la maschera interpolata vale 1 (tutti i nodi attorno sono liberi)
inline bool libero(const Mappa& mp, Punto p) { return interpola(mp, mp.mask, p) > 0.999; }

// Vero se il nodo (j, i, k) e' nella griglia e u vi e' finita (falso nei nodi ostacolo, che dopo il calcolo restano a INF)
inline bool finito(const Mappa& mp, int j, int i, int k)
{
    if (j < 0 || j >= mp.N || i < 0 || i >= mp.N || k < 0 || k >= mp.N) return false;
    return mp.u[(k * mp.N + i) * mp.N + j] < INF;
}

// Derivata di u nel nodo (j, i, k) lungo un asse; (dj, di, dk) = un passo lungo quell'asse, per esempio (1, 0, 0) per x.
// Differenza centrata se i due vicini sono liberi; vicino a un ostacolo si usa solo il vicino libero; 0 se non ce n'e' nessuno
inline double derivata(const Mappa& mp, int j, int i, int k, int dj, int di, int dk)
{
    const int N = mp.N;
    const bool su = finito(mp, j + dj, i + di, k + dk), giu = finito(mp, j - dj, i - di, k - dk);
    const double u0 = mp.u[(k * N + i) * N + j];
    const double us = su  ? mp.u[((k + dk) * N + i + di) * N + j + dj] : 0.0;   // u nel vicino "dopo"
    const double ug = giu ? mp.u[((k - dk) * N + i - di) * N + j - dj] : 0.0;   // u nel vicino "prima"
    if (su && giu) return (us - ug) / (2.0 * mp.h);     // differenza centrata
    if (su)        return (us - u0) / mp.h;             // solo in avanti
    if (giu)       return (u0 - ug) / mp.h;             // solo all'indietro
    return 0.0;
}

// Gradiente di u nel punto p: il gradiente negli 8 nodi della cella che contiene p, interpolato con gli stessi pesi di interpola().
// I nodi ostacolo non contano: si sommano solo i nodi finiti e si divide per la somma dei loro pesi
inline Punto gradiente(const Mappa& mp, Punto p)
{
    const double qx = (p.x + 1.0) / mp.h, qy = (p.y + 1.0) / mp.h, qz = (p.z + 1.0) / mp.h;   // come in interpola()
    const int j0 = std::clamp((int)std::floor(qx), 0, mp.N - 2);
    const int i0 = std::clamp((int)std::floor(qy), 0, mp.N - 2);
    const int k0 = std::clamp((int)std::floor(qz), 0, mp.N - 2);
    const double wx = qx - j0, wy = qy - i0, wz = qz - k0;
    Punto g = { 0.0, 0.0, 0.0 };
    double peso_tot = 0.0;
    for (int dk = 0; dk <= 1; ++dk)
        for (int di = 0; di <= 1; ++di)
            for (int dj = 0; dj <= 1; ++dj) {
                const int j = j0 + dj, i = i0 + di, k = k0 + dk;
                if (!finito(mp, j, i, k)) continue;                     // nodo ostacolo: non contribuisce
                const double peso = (dj ? wx : 1 - wx) * (di ? wy : 1 - wy) * (dk ? wz : 1 - wz);
                const Punto gn = { derivata(mp, j, i, k, 1, 0, 0),      // gradiente nel nodo: d/dx, d/dy, d/dz
                                   derivata(mp, j, i, k, 0, 1, 0),
                                   derivata(mp, j, i, k, 0, 0, 1) };
                g = g + peso * gn;
                peso_tot += peso;
            }
    return peso_tot > 0.0 ? (1.0 / peso_tot) * g : Punto{ 0.0, 0.0, 0.0 };   // nessun nodo finito: gradiente nullo
}

// Direzione di discesa in p: il versore -grad u / |grad u|, dove u (il tempo per arrivare a B) scende piu' in fretta.
// Falso se il gradiente e' nullo (non si sa dove andare)
inline bool direzione(const Mappa& mp, Punto p, Punto& d)
{
    const Punto g = gradiente(mp, p);
    const double ng = norma(g);
    if (ng < 1e-12) return false;
    d = (-1.0 / ng) * g;
    return true;
}

// Percorso da p0 a B: si integra dp/ds = -grad u / |grad u| (l'equazione del raggio) con Runge-Kutta del 4 ordine a passo ds = h/2.
// Ci si ferma a meno di 2h da B e si chiude su B. Se il gradiente si annulla, se un passo entrerebbe in un ostacolo
// o se B non si raggiunge entro 20 N passi, il programma si ferma con un messaggio
inline std::vector<Punto> percorso(const Mappa& mp, Punto p0)
{
    const double ds = mp.h / 2.0;                            // passo di integrazione (lunghezza di ogni tratto)
    std::vector<Punto> P = { p0 };
    Punto p = p0;
    for (int passo = 0; passo < 20 * mp.N; ++passo) {        // limite di sicurezza sul numero di passi
        if (norma(p - B) < 2.0 * mp.h) {                     // vicino all'arrivo: si chiude su B
            P.push_back(B);
            return P;
        }
        Punto k1, k2, k3, k4;                                // le quattro direzioni di RK4
        if (!direzione(mp, p, k1) ||
            !direzione(mp, p + (ds / 2.0) * k1, k2) ||
            !direzione(mp, p + (ds / 2.0) * k2, k3) ||
            !direzione(mp, p + ds * k3, k4)) {
            printf("RK4: gradiente nullo in (%.3f, %.3f, %.3f), il percorso non prosegue\n", p.x, p.y, p.z);
            exit(1);
        }
        const Punto q = p + (ds / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4);   // passo RK4: media pesata delle quattro direzioni
        if (!libero(mp, q)) {
            printf("RK4: il passo da (%.3f, %.3f, %.3f) entrerebbe in un ostacolo\n", p.x, p.y, p.z);
            exit(1);
        }
        P.push_back(q);
        p = q;
    }
    printf("RK4: B non raggiunto in %d passi (ultimo punto (%.3f, %.3f, %.3f))\n", 20 * mp.N, p.x, p.y, p.z);
    exit(1);
}

// VOLO
const double METRI_PER_UNITA = 10.0;                     // il cubo [-1, 1]^3 e' una stanza di 20 m di lato
const double V_MAX = 2.0;                                // velocita' massima del drone [m/s], dove F = 1
const double SEC_PER_U = METRI_PER_UNITA / V_MAX;        // 1 unita' di u = 5 s di volo
const double X_CAMBIO[N_MAPPE] = { 0.0, -0.54, -0.16 };     // la mappa m (m = 1, 2) arriva quando il drone supera x = X_CAMBIO[m]
                                                         // (poco prima della porta m); X_CAMBIO[0] non si usa: la mappa 0 vale alla partenza

// Un percorso con l'orario: il drone parte all'istante t0 e passa dal punto P[i] all'istante t[i] [s] con velocita' v[i] [m/s]
struct Piano {
    double t0;
    std::vector<Punto> P;
    std::vector<double> t, v;
};

// Istanti e velocita' lungo il percorso P, partendo all'istante t0: ogni tratto dura F (nel suo punto medio) x lunghezza x SEC_PER_U,
// cioe' u in secondi; la velocita' in ogni punto e' v = V_MAX / F. I passi di RK4 sono gia' corti (h/2): non serve ricampionare
inline Piano orario(const Mappa& mp, const std::vector<Punto>& P, double t0)
{
    Piano pi{ t0, P, { t0 }, {} };
    for (size_t i = 1; i < P.size(); ++i) {
        const Punto medio = 0.5 * (P[i - 1] + P[i]);
        pi.t.push_back(pi.t.back() + interpola(mp, mp.F, medio) * norma(P[i] - P[i - 1]) * SEC_PER_U);
    }
    for (const Punto& p : P) pi.v.push_back(V_MAX / interpola(mp, mp.F, p));
    return pi;
}

// Tutto il volo di un drone, per la grafica
struct Volo {
    std::vector<double> T, V;                                // istanti [s] e velocita' [m/s] dei punti volati
    std::vector<Punto> P;                                    // i punti volati
    std::vector<Piano> piani;                                // ogni percorso calcolato (alla partenza e dopo ogni cambio)
    std::vector<double> cambi;                               // istanti in cui la mappa cambia
    std::vector<int> ricalcolato;                            // per ogni cambio: 1 = ha ricalcolato, 0 = cambio lontano, nessun ricalcolo
    std::vector<std::pair<double, double>> ricalcolo, fermo; // intervalli [inizio, fine]: ricalcolo del FM2, sosta davanti alla porta
    void aggiungi(double t, Punto p, double v) { T.push_back(t); P.push_back(p); V.push_back(v); }
};

// Mentre ricalcola, il drone continua sul percorso vecchio pi dal punto k (dove la mappa e' cambiata): passa ancora dalla porta chiusa.
// Se il prossimo punto e' nel nuovo ostacolo frena e resta fermo fino a t_pronto, quando il percorso nuovo e' pronto.
// Restituisce vero se il drone arriva in B prima che il percorso nuovo sia pronto
inline bool durante_il_ricalcolo(const Piano& pi, size_t k, double t_pronto, const Mappa& nuova, Volo& v)
{
    double tl = pi.t[k], vl = pi.v[k];                       // istante, punto e velocita' dell'ultimo punto volato
    Punto pl = pi.P[k];
    for (size_t i = k + 1; i < pi.P.size(); ++i) {
        if (pi.t[i] >= t_pronto) {                           // il percorso nuovo e' pronto prima di arrivare al punto i:
            const double a = (t_pronto - tl) / (pi.t[i] - tl);   // il drone ha fatto la frazione a del tratto
            v.aggiungi(t_pronto, pl + a * (pi.P[i] - pl), vl + a * (pi.v[i] - vl));
            return false;
        }
        if (!libero(nuova, pi.P[i])) {                       // il punto i e' nel nuovo ostacolo: il drone si ferma dov'e'
            v.aggiungi(std::min(tl + 1e-3, t_pronto), pl, 0.0);
            v.aggiungi(t_pronto, pl, 0.0);
            v.fermo.push_back({ tl, t_pronto });
            return false;
        }
        v.aggiungi(pi.t[i], pi.P[i], pi.v[i]);
        tl = pi.t[i]; pl = pi.P[i]; vl = pi.v[i];
    }
    return true;                                             // arrivato in B mentre ricalcolava
}

// Distanza del punto p dal parallelepipedo z (0 se p e' dentro)
inline double distanza(const Zona& z, Punto p)
{
    const double dx = std::max({ z.x0 - p.x, 0.0, p.x - z.x1 });
    const double dy = std::max({ z.y0 - p.y, 0.0, p.y - z.y1 });
    const double dz = std::max({ z.z0 - p.z, 0.0, p.z - z.z1 });
    return std::sqrt(dx * dx + dy * dy + dz * dz);
}

// Vero se un punto del percorso dal punto k in poi (la parte ancora da volare) e' vicino a uno degli oggetti, cioe' entro
// D_SICUREZZA (+ un passo h di sicurezza): oltre questa distanza l'oggetto non cambia F, quindi nemmeno il percorso
inline bool vicino(const Piano& pi, size_t k, const std::vector<Zona>& oggetti, double h)
{
    const double soglia = D_SICUREZZA + h;
    for (size_t i = k; i < pi.P.size(); ++i)
        for (const Zona& z : oggetti)
            if (distanza(z, pi.P[i]) < soglia) return true;
    return false;
}

// Volo da A a B. Alla partenza vale la mappa 0; la mappa m (m = 1, 2) arriva quando il drone supera x = X_CAMBIO[m].
// A ogni cambio il drone ricalcola solo se serve: se e' stato tolto un oggetto (potrebbe aprirsi una scorciatoia), oppure se
// un oggetto aggiunto e' vicino al percorso che resta. Se ricalcola, lo fa per t_ricalcolo[m] secondi (il tempo misurato del
// suo metodo) continuando sul percorso vecchio, poi riparte sulla mappa nuova dal punto in cui si trova. Altrimenti continua
// sul percorso che sta seguendo: resta il migliore anche nella mappa nuova
inline Volo vola(const std::vector<Mappa>& mappe, const std::vector<double>& t_ricalcolo)
{
    Volo v;
    Punto pos = A;                                           // da dove parte il percorso attuale, quando e su quale mappa
    double t = 0.0;
    int m = 0;
    while (true) {
        const Piano pi = orario(mappe[m], percorso(mappe[m], pos), t);
        v.piani.push_back(pi);
        if (v.P.empty()) v.aggiungi(0.0, A, pi.v[0]);         // il punto di partenza
        size_t da = 0;                                       // ultimo punto gia' volato di questo percorso
        while (true) {                                       // vola su questo percorso, un cambio di mappa alla volta
            const int nuova = m + 1;                         // la prossima mappa (se c'e')
            size_t k = 0;                                    // primo punto oltre X_CAMBIO[nuova]; 0 = nessuno
            if (nuova < (int)mappe.size())
                for (size_t i = da; i < pi.P.size() && k == 0; ++i)
                    if (pi.P[i].x > X_CAMBIO[nuova]) k = std::max<size_t>(i, 1);
            if (k == 0) {                                    // nessun altro cambio: vola fino a B
                for (size_t i = da + 1; i < pi.P.size(); ++i) v.aggiungi(pi.t[i], pi.P[i], pi.v[i]);
                return v;
            }
            for (size_t i = da + 1; i <= k; ++i) v.aggiungi(pi.t[i], pi.P[i], pi.v[i]);   // vola fino al punto del cambio
            v.cambi.push_back(pi.t[k]);
            m = nuova;                                       // da qui vale la mappa nuova
            const bool serve = !mappe[m].tolti.empty() || vicino(pi, k, mappe[m].aggiunti, mappe[m].h);
            v.ricalcolato.push_back(serve ? 1 : 0);
            if (!serve) { da = k; continue; }                // cambio lontano: continua sullo stesso percorso
            const double t_pronto = pi.t[k] + t_ricalcolo[m];                           // quando il percorso nuovo e' pronto
            v.ricalcolo.push_back({ pi.t[k], t_pronto });
            if (durante_il_ricalcolo(pi, k, t_pronto, mappe[m], v)) return v;          // arrivato in B mentre ricalcolava
            pos = v.P.back(); t = t_pronto;                                             // riparte da dove si trova, sulla mappa nuova
            break;
        }
    }
}
// File dei voli per la cella Python della grafica: una riga per ogni dato, con un'etichetta all'inizio
inline void scrivi_voli(const std::string& nome, double uA, const std::vector<std::string>& metodi, const std::vector<Volo>& voli)
{
    FILE* f = fopen(nome.c_str(), "w");
    fprintf(f, "PARAMETRI %.17g %.17g %.17g %.17g %.17g\n", METRI_PER_UNITA, V_MAX, SEC_PER_U, V_MIN, D_SICUREZZA);
    fprintf(f, "UA %.9g\n", uA);                             // u(A) con le porte aperte
    for (size_t n = 0; n < voli.size(); ++n) {
        const Volo& v = voli[n];
        fprintf(f, "METODO %s\n", metodi[n].c_str());
        for (size_t i = 0; i < v.T.size(); ++i)              // i punti volati: t x y z v
            fprintf(f, "PUNTO %.9g %.9g %.9g %.9g %.9g\n", v.T[i], v.P[i].x, v.P[i].y, v.P[i].z, v.V[i]);
        for (const Piano& pi : v.piani) {                    // i percorsi calcolati: t0, poi t x y z di ogni punto
            fprintf(f, "PIANO %.9g\n", pi.t0);
            for (size_t i = 0; i < pi.P.size(); ++i)
                fprintf(f, "PP %.9g %.9g %.9g %.9g\n", pi.t[i], pi.P[i].x, pi.P[i].y, pi.P[i].z);
        }
        for (size_t i = 0; i < v.cambi.size(); ++i) fprintf(f, "CAMBIO %.9g %d\n", v.cambi[i], v.ricalcolato[i]);   // istante, 1 = ha ricalcolato
        for (auto [a, b] : v.ricalcolo) fprintf(f, "RICALCOLO %.9g %.9g\n", a, b);
        for (auto [a, b] : v.fermo) fprintf(f, "FERMO %.9g %.9g\n", a, b);
    }
    fclose(f);
}

In [ ]:
%%writefile main.cu
// FM2 (passata 1 + passata 2) con il FMM, il FIM-Jacobi e il Block FIM: tabella al variare di N e ripianificazione nel labirinto
#include <cstdio>
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include <string>          // nomi dei file delle mappe
#include "fmm3d.h"         // Fast Marching (CPU)
#include "ambiente.h"      // stanza, ostacoli, maschera, lentezza del FM2
#include "block_fim3d.cuh" // Block FIM (GPU)
#include "jacobi_fim.cuh"  // FIM-Jacobi (GPU)
#include "percorso.h"      // percorso del drone (RK4) e volo con i cambi di mappa


// FM2 completo (passata 1 + passata 2) sulla mappa attuale, con il FMM (CPU), il FIM-Jacobi (GPU) o il Block FIM (GPU).
enum Metodo { FMM, FIM_JACOBI, BLOCK_FIM };                  // con quale metodo si risolvono le due passate
int fm2(Metodo metodo, int Nk, const std::vector<unsigned char>& mask, float eps, int niter,
        std::vector<float>& u, std::vector<float>& Fk, double& t_ms, double* t1_ms = nullptr, double* ttot_ms = nullptr)
{
    const auto t_inizio = std::chrono::high_resolution_clock::now();   // il tempo totale parte da qui
    const float hk = 2.0f / (Nk - 1);
    const int n = Nk * Nk * Nk;
    // prima passata: le sorgenti sono i nodi liberi che toccano un ostacolo
    const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
    // seconda passata: la sorgente e' B
    const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };
    std::vector<float> d(n, INF);
    for (int s : bordo) d[s] = 0.0f;
    const std::vector<float> uno(n, 1.0f);
    u.assign(n, INF);
    for (int s : srcs) u[s] = 0.0f;
    int giri = 0;
    if (metodo == FMM) {                                     // CPU: le due chiamate al FMM
        auto t0 = std::chrono::high_resolution_clock::now();
        fmm(d, uno, Nk, hk, bordo, mask);
        auto t1 = std::chrono::high_resolution_clock::now();
        Fk = lentezza_fm2(d);
        auto t2 = std::chrono::high_resolution_clock::now();
        fmm(u, Fk, Nk, hk, srcs, mask);
        auto t3 = std::chrono::high_resolution_clock::now();
        const double ms1 = std::chrono::duration<double, std::milli>(t1 - t0).count();    // passata 1
        t_ms = ms1 + std::chrono::duration<double, std::milli>(t3 - t2).count();         // le due passate (senza lentezza_fm2, come sulla GPU)
        if (t1_ms) *t1_ms = ms1;
    } else {                                                 // GPU (FIM-Jacobi o Block FIM): i dati restano sulla GPU
        const size_t bytes = (size_t)n * sizeof(float);
        float *d_d, *d_F, *d_u; unsigned char* d_mask;
        CUDA_CHECK(cudaMalloc(&d_d, bytes)); CUDA_CHECK(cudaMalloc(&d_F, bytes));
        CUDA_CHECK(cudaMalloc(&d_u, bytes)); CUDA_CHECK(cudaMalloc(&d_mask, n));
        // CPU -> GPU, una volta sola per tutto il FM2
        CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), n, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_d, d.data(), bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_F, uno.data(), bytes, cudaMemcpyHostToDevice));   // F = 1 per la passata 1
        CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
        float ms1 = 0.0f, ms2 = 0.0f;
        // passata 1: distanza d dagli ostacoli
        if (metodo == BLOCK_FIM) block_fim(d_d, d_F, d_mask, Nk, hk, bordo, eps, niter, &ms1);
        else                     jacobi_fim(d_d, d_F, d_mask, Nk, hk, eps, &ms1);
        // lentezza sulla GPU: F prende il posto degli "uno" nello stesso array; d non torna sulla CPU
        const int threads = 256;
        lentezza_fm2_gpu<<<(n + threads - 1) / threads, threads>>>(d_d, d_F, n, V_MIN, D_SICUREZZA);
        CUDA_CHECK(cudaGetLastError());
        // passata 2: tempo di arrivo da B
        if (metodo == BLOCK_FIM) giri = block_fim(d_u, d_F, d_mask, Nk, hk, srcs, eps, niter, &ms2);
        else                     giri = jacobi_fim(d_u, d_F, d_mask, Nk, hk, eps, &ms2);
        // GPU -> CPU: u e F (servono al percorso del drone), poi si libera la memoria
        Fk.resize(n);
        CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaMemcpy(Fk.data(), d_F, bytes, cudaMemcpyDeviceToHost));
        CUDA_CHECK(cudaFree(d_d)); CUDA_CHECK(cudaFree(d_F)); CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_mask));
        t_ms = ms1 + ms2;                                    // solo i cicli di calcolo, come prima
        if (t1_ms) *t1_ms = ms1;
    }
    const auto t_fine = std::chrono::high_resolution_clock::now();
    if (ttot_ms) *ttot_ms = std::chrono::duration<double, std::milli>(t_fine - t_inizio).count();
    return giri;
}

// Scrive l'ambiente della mappa m per la grafica in Python: ostacoli e fori della mappa attuale, A, B
void scrivi_ambiente(int m)
{
    std::ofstream amb("ambiente_mappa" + std::to_string(m) + ".txt");
    amb.precision(15);
    amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
    amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
    for (const auto& os : OSTACOLI) amb << "ZONA " << os.x0 << " " << os.x1 << " " << os.y0 << " " << os.y1 << " " << os.z0 << " " << os.z1 << "\n";
    for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
}

int main()
{
    // Informazioni sulla GPU usata
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 8;                                     // iterazioni locali massime (uscita anticipata): la migliore con gli ostacoli (prove 12 / 24)

    if (!ambiente_valido()) {                                // A o B dentro un ostacolo: il problema non ha senso
        printf("Errore: A o B sono dentro un ostacolo, correggi ambiente.h\n");
        return 1;
    }
    // Al variare di N: i risultati si salvano nel ciclo e si stampano dopo, in tre tabelle con le stesse righe (accuratezza,
    // tempo di calcolo, tempo totale), cosi' i due tempi si confrontano uno sopra l'altro
    std::vector<int> N_tab; std::vector<float> uA_tab, diff_fmm_tab, diff_jfim_tab;
    std::vector<double> t_fmm_tab, t_jfim_tab, t_bfim_tab, ttot_fmm, ttot_jac, ttot_bfim;
    for (int Nk : {256, 384}) {
        const std::vector<unsigned char> mask = costruisci_maschera(Nk);    // 1 = libero, 0 = ostacolo
        const int iA = nodo(Nk, A.x, A.y, A.z);              // nodo della partenza A
        // FM2 completo (passata 1 + passata 2) con i tre metodi: ognuno usa la propria F
        std::vector<float> u_fmm, u_bfim, u_jac, F_fmm, F_bfim, F_jac;
        double t_fmm = 0.0, t_bfim = 0.0, t_jac = 0.0, tt_fmm = 0.0, tt_jac = 0.0, tt_bfim = 0.0;
        fm2(FMM, Nk, mask, eps, niter, u_fmm, F_fmm, t_fmm, nullptr, &tt_fmm);
        fm2(FIM_JACOBI, Nk, mask, eps, niter, u_jac, F_jac, t_jac, nullptr, &tt_jac);
        fm2(BLOCK_FIM, Nk, mask, eps, niter, u_bfim, F_bfim, t_bfim, nullptr, &tt_bfim);
        N_tab.push_back(Nk);
        ttot_fmm.push_back(tt_fmm); ttot_jac.push_back(tt_jac); ttot_bfim.push_back(tt_bfim); 

        // Controlli: stessa u dei tre metodi, nessun NaN, ostacoli a INF
        float diff = 0.0f, diff_jac = 0.0f;
        int n_nan = 0, ost_non_inf = 0;
        for (size_t q = 0; q < u_bfim.size(); ++q) {
            diff = fmaxf(diff, fabsf(u_fmm[q] - u_bfim[q]));
            diff_jac = fmaxf(diff_jac, fabsf(u_jac[q] - u_bfim[q]));
            if (std::isnan(u_bfim[q])) ++n_nan;
            if (!mask[q] && u_bfim[q] < INF) ++ost_non_inf;
        }
        if (n_nan > 0 || ost_non_inf > 0)                    // si stampa solo se qualcosa non va
            printf("ATTENZIONE (N = %d): %d valori NaN, %d nodi ostacolo raggiunti dall'onda\n", Nk, n_nan, ost_non_inf);

        uA_tab.push_back(u_bfim[iA]); diff_fmm_tab.push_back(diff); diff_jfim_tab.push_back(diff_jac);
        t_fmm_tab.push_back(t_fmm); t_jfim_tab.push_back(t_jac); t_bfim_tab.push_back(t_bfim);
    }
    // 1) Accuratezza: la u dei tre metodi coincide?
    printf("Accuratezza: max|...| = differenza massima dalla u del Block FIM su tutta la griglia; u(A) = tempo di arrivo da A\n");
    printf("    N   max|FMM-BFIM|   max|J-FIM-BFIM|   u(A) BFIM\n");
    for (size_t r = 0; r < N_tab.size(); ++r) {
        if (uA_tab[r] >= INF) printf("%5d   A NON RAGGIUNGIBILE da B\n", N_tab[r]);   // l'onda partita da B non e' mai arrivata ad A
        else printf("%5d   %13.2e   %15.2e   %9.4f\n", N_tab[r], diff_fmm_tab[r], diff_jfim_tab[r], uA_tab[r]);
    }
    // 2) Tempo di calcolo: solo i cicli di calcolo delle due passate (come nel codice di Jeong)
    printf("\nTempo di calcolo [ms]: solo i cicli di calcolo delle due passate; speedup = t metodo / t Block FIM\n");
    printf("    N   t FMM [ms]   t J-FIM [ms]   t BFIM [ms]   sp vs FMM   sp vs J-FIM\n");
    for (size_t r = 0; r < N_tab.size(); ++r)
        printf("%5d   %10.1f   %12.2f   %11.2f   %9.1f   %11.1f\n", N_tab[r], t_fmm_tab[r], t_jfim_tab[r], t_bfim_tab[r],
               t_fmm_tab[r] / t_bfim_tab[r], t_jfim_tab[r] / t_bfim_tab[r]);
    // 3) Tempo totale, stesse righe: anche allocazione, copie CPU <-> GPU e lentezza (quello che aspetterebbe davvero un drone)
    printf("\nTempo totale [ms]: dalla maschera pronta sulla CPU a u e F di nuovo sulla CPU (allocazione, copie, calcolo e lentezza)\n");
    printf("    N   t FMM [ms]   t J-FIM [ms]   t BFIM [ms]   sp vs FMM   sp vs J-FIM\n");
    for (size_t r = 0; r < N_tab.size(); ++r)
        printf("%5d   %10.1f   %12.2f   %11.2f   %9.1f   %11.1f\n", N_tab[r], ttot_fmm[r], ttot_jac[r], ttot_bfim[r],
               ttot_fmm[r] / ttot_bfim[r], ttot_jac[r] / ttot_bfim[r]);
    // RIPIANIFICAZIONE
    {
        const int Nr = 384;                                  // griglia della ripianificazione 
        std::vector<Mappa> mappe;                            // le mappe risolte con il Block FIM: i droni volano su queste
        std::vector<double> t_ric_fmm(N_MAPPE), t_ric_jac(N_MAPPE), t_ric_bfim(N_MAPPE);     // tempo di ricalcolo [s] di ogni mappa, per metodo
        const char* nomi[N_MAPPE] = { "0 porte aperte", "1 porta 1 chiusa", "2 porte 1 e 2 chiuse" };
        // Per ogni mappa: tempi totali e accuratezza, stampati dopo il ciclo in due tabelle (prima i tempi, poi l'accuratezza)
        std::vector<double> tr_fmm(N_MAPPE), tr_jac(N_MAPPE), tr_bfim(N_MAPPE);
        std::vector<float> uA_r(N_MAPPE), diff_r(N_MAPPE), diffj_r(N_MAPPE);
        for (int m = 0; m < N_MAPPE; ++m) {
            std::vector<Zona> aggiunti, tolti;               // cosa cambia rispetto alla mappa precedente
            imposta_mappa(m, aggiunti, tolti);
            const std::vector<unsigned char> mask = costruisci_maschera(Nr);
            const int iA = nodo(Nr, A.x, A.y, A.z);
            std::vector<float> u_fmm, u_bfim, u_jac, F_fmm, F_bfim, F_jac;
            double t_fmm = 0.0, t_bfim = 0.0, t_jac = 0.0, tt_fmm = 0.0, tt_jac = 0.0, tt_bfim = 0.0;
            fm2(FMM, Nr, mask, eps, niter, u_fmm, F_fmm, t_fmm, nullptr, &tt_fmm);
            fm2(FIM_JACOBI,  Nr, mask, eps, niter, u_jac, F_jac, t_jac, nullptr, &tt_jac);
            fm2(BLOCK_FIM, Nr, mask, eps, niter, u_bfim, F_bfim, t_bfim, nullptr, &tt_bfim);
            t_ric_fmm[m] = tt_fmm / 1000.0; t_ric_jac[m] = tt_jac / 1000.0; t_ric_bfim[m] = tt_bfim / 1000.0;   // il drone aspetta il tempo totale
            float diff = 0.0f, diff_jac = 0.0f;
            for (size_t q = 0; q < u_bfim.size(); ++q) {
                diff = fmaxf(diff, fabsf(u_fmm[q] - u_bfim[q]));
                diff_jac = fmaxf(diff_jac, fabsf(u_jac[q] - u_bfim[q]));
            }
            tr_fmm[m] = tt_fmm; tr_jac[m] = tt_jac; tr_bfim[m] = tt_bfim;
            uA_r[m] = u_bfim[iA]; diff_r[m] = diff; diffj_r[m] = diff_jac;
            scrivi_ambiente(m);
            mappe.push_back({ Nr, 2.0 / (Nr - 1), std::move(u_bfim), std::move(F_bfim), mask, aggiunti, tolti });
        }
        imposta_mappa(0);
        // 1) Tempi totali: e' il tempo che il drone aspetta nel volo
        printf("\nRipianificazione (N = %d): tempo TOTALE per ricalcolare il FM2 dopo ogni cambio di mappa (allocazione, copie, calcolo e\n", Nr);
        printf("lentezza comprese): e' il tempo che il drone aspetta nel volo\n");
        printf("  mappa                        t FMM [ms]   t J-FIM [ms]   t BFIM [ms]   sp vs FMM   sp vs J-FIM\n");
        for (int m = 0; m < N_MAPPE; ++m)
            printf("  %-26s   %10.1f   %12.2f   %11.2f   %9.1f   %11.1f\n", nomi[m], tr_fmm[m], tr_jac[m], tr_bfim[m],
                   tr_fmm[m] / tr_bfim[m], tr_jac[m] / tr_bfim[m]);
        // 2) Accuratezza: la u dei tre metodi coincide in ogni mappa?
        printf("\nRipianificazione (N = %d): accuratezza (max|...| = differenza massima dalla u del Block FIM su tutta la griglia)\n", Nr);
        printf("  mappa                        u(A) BFIM   max|FMM-BFIM|   max|J-FIM-BFIM|\n");
        for (int m = 0; m < N_MAPPE; ++m)
            printf("  %-26s   %9.4f   %13.2e   %15.2e\n", nomi[m], uA_r[m], diff_r[m], diffj_r[m]);
        // VOLO (percorso.h): i tre droni volano sulla stessa u (del Block FIM); a ogni cambio ognuno aspetta il tempo totale del suo metodo
        const Volo volo_fmm = vola(mappe, t_ric_fmm), volo_jac = vola(mappe, t_ric_jac), volo_bfim = vola(mappe, t_ric_bfim);
        scrivi_voli("voli_" + std::to_string(Nr) + ".txt", mappe[0].u[nodo(Nr, A.x, A.y, A.z)],
                    { "FMM", "FIM-Jacobi", "Block FIM" }, { volo_fmm, volo_jac, volo_bfim });
    }
    return 0;
}

In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_75 main.cu -o fim3d && ./fim3d

In [ ]:
# RIPIANIFICAZIONE NEL LABIRINTO - GRAFICA: animazione 3D dei droni e grafico delle velocita' (usa i voli letti nella cella sopra).
# Si cambia solo per l'aspetto: colori, dimensioni, velocita' dell'animazione.
import glob
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


N = 384                                                             # griglia della ripianificazione (Nr in main.cu)
METODI = ["FMM", "FIM-Jacobi", "Block FIM"]                                # da sinistra a destra

def leggi_ambiente(nome):
    amb = {"ZONA": [], "FORO": []}
    for riga in open(nome):
        k, *v = riga.split(); v = [float(t) for t in v]
        if k in ("A", "B"): amb[k] = np.array(v)
        else: amb[k].append(tuple(v))
    return amb
ambienti = [leggi_ambiente(f"ambiente_mappa{m}.txt") for m in range(len(glob.glob("ambiente_mappa*.txt")))]   # ostacoli e fori di ogni mappa
A, B = ambienti[0]["A"], ambienti[0]["B"]

voli = {}                                                           # per metodo: punti volati, percorsi calcolati, cambi, ricalcoli, soste
for riga in open(f"voli_{N}.txt"):
    k, *v = riga.split()
    if k == "METODO":
        met = " ".join(v); voli[met] = dict(T=[], P=[], V=[], piani=[], cambi=[], ricalcolato=[], ricalcolo=[], fermo=[]); continue
    v = [float(t) for t in v]
    if k == "PARAMETRI": METRI_PER_UNITA, V_MAX, SEC_PER_U, V_MIN, D_SICUREZZA = v
    elif k == "UA": uA = v[0]                                        # u(A) con le porte aperte
    elif k == "PUNTO": voli[met]["T"].append(v[0]); voli[met]["P"].append(v[1:4]); voli[met]["V"].append(v[4])
    elif k == "PIANO": voli[met]["piani"].append((v[0], [], []))
    elif k == "PP": voli[met]["piani"][-1][1].append(v[1:4]); voli[met]["piani"][-1][2].append(v[0])
    elif k == "CAMBIO": voli[met]["cambi"].append(v[0]); voli[met]["ricalcolato"].append(v[1] == 1)   # istante; ha ricalcolato?
    else: voli[met][k.lower()].append(tuple(v))                     # RICALCOLO e FERMO: intervalli (inizio, fine)
for v in voli.values():
    v["T"], v["P"], v["V"] = np.array(v["T"]), np.array(v["P"]), np.array(v["V"])
    v["piani"] = [(t0, np.array(Pp), np.array(tp)) for t0, Pp, tp in v["piani"]]
print(f"volo con le porte aperte: {uA * SEC_PER_U:.2f} s")                 # u(A) in secondi
for met in METODI:
    print(f"{met}: arrivo in B dopo {voli[met]['T'][-1]:.2f} s")
V_PIENA = 0.95 * V_MAX                                              # "velocita' massima": almeno il 95% di V_MAX

# 4) PARAMETRI DELLA GRAFICA E GEOMETRIA DA DISEGNARE
DT_FRAME = 0.3                                                      # un fotogramma ogni 0.3 s di volo
MS_FRAME = 60                                                       # il Play mostra un fotogramma ogni 60 ms: animazione 0.3 / 0.06 = 5 volte piu' veloce del vero
GRIGIO, ARANCIO, BLU, ROSSO = "#52514e", "#eb6834", "#2a78d6", "#c0392b"   # FMM, FIM-Jacobi, Block FIM (come nella stanza); porte chiuse
colore = {"FMM": GRIGIO, "FIM-Jacobi": ARANCIO, "Block FIM": BLU}
chiaro = {"FMM": "#b9b8b4", "FIM-Jacobi": "#f7c3a8", "Block FIM": "#a9cbf0"}   # sfumatura chiara dello stesso colore (lampeggio)
scena_di = {met: "scene" if i == 0 else f"scene{i + 1}" for i, met in enumerate(METODI)}   # una scena per drone, da sinistra
DIM_DRONE = 4                                                       # diametro del pallino del drone e dei punti A e B [pixel]
AVVISO = 2.0                                                        # per quanti secondi di volo resta la scritta "cambio lontano: nessun ricalcolo"

def mesh(ostacoli, nome, col, opacita):                              # un solo Mesh3d con tutti i parallelepipedi
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for b in ostacoli:
        o = len(X)
        X += [b[0], b[1], b[1], b[0], b[0], b[1], b[1], b[0]]; Y += [b[2], b[2], b[3], b[3]] * 2; Z += [b[4]] * 4 + [b[5]] * 4
        for i, j, k in [(0, 1, 2), (0, 2, 3), (4, 5, 6), (4, 6, 7), (0, 1, 5), (0, 5, 4), (1, 2, 6), (1, 6, 5), (2, 3, 7), (2, 7, 6), (3, 0, 4), (3, 4, 7)]:
            I.append(o + i); J.append(o + j); K.append(o + k)
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color=col, opacity=opacita, name=nome, flatshading=True, hoverinfo="name")
a0 = ambienti[0]
cambi_mappa = []                                                    # per ogni cambio m = 1, 2, ...: gli oggetti che compaiono, con il nome
n_porta = 0
for prima, dopo in zip(ambienti[:-1], ambienti[1:]):
    nuovi = []
    for f in prima["FORO"]:
        if f not in dopo["FORO"]:                                   # un foro che sparisce: una porta che si chiude
            n_porta += 1; nuovi.append((f, f"porta {n_porta}"))
    nuovi += [(z, "oggetto nuovo") for z in dopo["ZONA"] if z not in prima["ZONA"]]   # un ostacolo che compare (la colonna)
    cambi_mappa.append(nuovi)
interni = [b for b in a0["ZONA"] if b[0] > -0.99 and b[1] < 0.99]   # i muri interni (le pareti della stanza no: c'e' il contorno)
fori_di = lambda b: [f for f in a0["FORO"] if b[0] - 1e-9 <= f[0] and f[1] <= b[1] + 1e-9]   # le aperture di quel muro
def superficie(b, fori, nome, col, opacita):                       # solo le facce esterne del muro b con i suoi fori (niente facce interne)
    tagli = [sorted({b[2 * a], b[2 * a + 1]} | {min(max(f[2 * a + s], b[2 * a]), b[2 * a + 1]) for f in fori for s in (0, 1)}) for a in range(3)]
    n = [len(t) - 1 for t in tagli]
    pieno = np.zeros(n, bool)                                       # cella (ix, iy, iz) del muro: piena se non sta in un foro
    for ix in range(n[0]):
        for iy in range(n[1]):
            for iz in range(n[2]):
                c = [(tagli[a][i] + tagli[a][i + 1]) / 2 for a, i in enumerate((ix, iy, iz))]
                pieno[ix, iy, iz] = not any(f[0] <= c[0] <= f[1] and f[2] <= c[1] <= f[3] and f[4] <= c[2] <= f[5] for f in fori)
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for ix, iy, iz in zip(*np.nonzero(pieno)):
        for asse in range(3):
            for verso in (0, 1):                                    # faccia verso il basso (0) o verso l'alto (1) lungo l'asse
                vic = [ix, iy, iz]; vic[asse] += 1 if verso else -1
                if 0 <= vic[asse] < n[asse] and pieno[tuple(vic)]: continue   # confina con un'altra cella piena: faccia interna
                lo = [tagli[a][i] for a, i in enumerate((ix, iy, iz))]; hi = [tagli[a][i + 1] for a, i in enumerate((ix, iy, iz))]
                q = lo[asse] if verso == 0 else hi[asse]
                u_, w_ = [a for a in range(3) if a != asse]
                o = len(X)
                for du, dw in [(0, 0), (1, 0), (1, 1), (0, 1)]:
                    v = [0.0] * 3; v[asse] = q; v[u_] = (lo[u_], hi[u_])[du]; v[w_] = (lo[w_], hi[w_])[dw]
                    X.append(v[0]); Y.append(v[1]); Z.append(v[2])
                I += [o, o]; J += [o + 1, o + 2]; K += [o + 2, o + 3]
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color=col, opacity=opacita, name=nome, flatshading=True, hoverinfo="name")
def muri_mesh():                                                    # tutti i muri interni in un solo Mesh3d
    parti = [superficie(b, fori_di(b), "muri", "#9a9a9a", 0.35) for b in interni]
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for m_ in parti:
        o = len(X); X += list(m_.x); Y += list(m_.y); Z += list(m_.z)
        I += [o + i for i in m_.i]; J += [o + j for j in m_.j]; K += [o + k for k in m_.k]
    return go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color="#9a9a9a", opacity=0.35, name="muri", flatshading=True, hoverinfo="name")
spigoli = []                                                        # i 12 spigoli della stanza
for a in (-1, 1):
    for b in (-1, 1):
        spigoli += [[(-1, a, b), (1, a, b)], [(a, -1, b), (a, 1, b)], [(a, b, -1), (a, b, 1)]]
stanza = go.Scatter3d(x=sum([[p[0][0], p[1][0], None] for p in spigoli], []), y=sum([[p[0][1], p[1][1], None] for p in spigoli], []),
                      z=sum([[p[0][2], p[1][2], None] for p in spigoli], []), mode="lines", line=dict(color="gray", width=2),
                      name="stanza (20 m)", hoverinfo="skip")
ab = go.Scatter3d(x=[A[0], B[0]], y=[A[1], B[1]], z=[A[2], B[2]], mode="markers+text", text=["  A (partenza)", "  B (arrivo)"],
                  marker=dict(size=DIM_DRONE, color=["green", "black"]), textfont=dict(size=14), name="A e B")
def centro(b):                                                      # dove scrivere il nome di un ostacolo: sopra il centro
    return [(b[0] + b[1]) / 2], [(b[2] + b[3]) / 2], [b[5] + 0.08]

# 5) ANIMAZIONE 3D: da sinistra i droni del FMM, del FIM-Jacobi e del Block FIM (stessi istanti)
def compare(b, t_ev, t, nome, scena):                               # oggetto che compare all'istante t_ev (porta chiusa o ostacolo nuovo; prima: vuoto)
    if t < t_ev:
        return [go.Mesh3d(x=[], y=[], z=[], i=[], j=[], k=[], name=nome, scene=scena, showlegend=False),
                go.Scatter3d(x=[], y=[], z=[], mode="text", name=nome, scene=scena, showlegend=False)]
    cx, cy, cz = centro(b)
    mm = mesh([b], nome, ROSSO, 0.85); mm.update(scene=scena, showlegend=False)
    return [mm, go.Scatter3d(x=cx, y=cy, z=cz, mode="text", text=[nome], textfont=dict(size=13, color=ROSSO), scene=scena, showlegend=False)]
def posizione(met, t):
    v = voli[met]; T, P, V = v["T"], v["P"], v["V"]
    if t >= T[-1]: return P[-1], 0.0
    k = np.searchsorted(T, t, side="right"); a = (t - T[k - 1]) / max(T[k] - T[k - 1], 1e-12)
    return P[k - 1] + a * (P[k] - P[k - 1]), V[k - 1] + a * (V[k] - V[k - 1])
def drone(met, t):                                                  # scia colorata con la velocita', percorso previsto e drone
    v = voli[met]; T, P, V = v["T"], v["P"], v["V"]
    pos, vel = posizione(met, t)
    fermo = any(a <= t < b for a, b in v["fermo"])
    ricalcola = any(a <= t < b for a, b in v["ricalcolo"])          # mentre ricalcola il pallino lampeggia
    piano = [p for p in v["piani"] if p[0] <= t][-1]
    resto = np.vstack([pos[None], piano[1][piano[2] > t]]) if not fermo and t < T[-1] else np.empty((0, 3))
    scia, vs = np.vstack([P[T <= t][::3], pos[None]]), np.append(V[T <= t][::3], vel)
    sc, c = scena_di[met], colore[met]
    return [go.Scatter3d(x=scia[:, 0], y=scia[:, 1], z=scia[:, 2], mode="lines", scene=sc, showlegend=False, name=f"scia {met}",
                         line=dict(color=vs, colorscale="Turbo", cmin=0.0, cmax=V_MAX, width=12,
                                   colorbar=dict(title="velocita'<br>[m/s]", x=1.0, len=0.6) if met == "Block FIM" else None,
                                   showscale=(met == "Block FIM"))),
            go.Scatter3d(x=resto[:, 0], y=resto[:, 1], z=resto[:, 2], mode="lines", scene=sc, showlegend=False,
                         line=dict(color="black", width=2, dash="dash"), name=f"percorso previsto {met}"),
            go.Scatter3d(x=[pos[0]], y=[pos[1]], z=[pos[2]], mode="markers", scene=sc, showlegend=False, name=f"drone {met}",
                         marker=dict(size=DIM_DRONE, color=chiaro[met] if ricalcola and round(t / DT_FRAME) % 2 else c,   # lampeggia: un fotogramma si' e uno no
                         line=dict(color=c, width=2)))]
def stato(met, t):
    v = voli[met]
    if t >= v["T"][-1]: return "arrivato"
    if any(a <= t < b for a, b in v["fermo"]): return "FERMO davanti alla porta, ricalcola"
    if any(a <= t < b for a, b in v["ricalcolo"]): return f"ricalcola, vola sul percorso vecchio a {posizione(met, t)[1]:.1f} m/s"
    if any(c <= t < c + AVVISO and not r for c, r in zip(v["cambi"], v["ricalcolato"])): return "cambio lontano: nessun ricalcolo"
    return f"in volo a {posizione(met, t)[1]:.1f} m/s"
ASSI = dict(range=[-1.0, 1.0], autorange=False, showticklabels=False, title="")
SCENA = dict(xaxis=ASSI, yaxis=ASSI, zaxis=ASSI, aspectmode="cube", camera=dict(eye=dict(x=1.2, y=-1.7, z=1.3)))
def statiche(scena):
    out = []
    for tr in [stanza, muri_mesh(), ab]:
        tr = go.Figure(tr).data[0]; tr.update(scene=scena, showlegend=False); out.append(tr)
    return out
def dati(t):
    out = []
    for met in METODI:
        sc = scena_di[met]
        c = voli[met]["cambi"] + [np.inf] * len(cambi_mappa)        # istante di ogni cambio per questo drone (inf: non ci arriva)
        for i, nuovi in enumerate(cambi_mappa):
            for b, nome in nuovi: out += compare(b, c[i], t, nome, sc)
        out += drone(met, t)
    return out
def titolo(t):
    return f"t = {t:4.1f} s   |   " + "   |   ".join(f"<span style='color:{colore[m]}'>{m}: {stato(m, t)}</span>" for m in METODI)
t_fine = max(v["T"][-1] for v in voli.values())
istanti = np.append(np.arange(0.0, t_fine, DT_FRAME), t_fine)
fisse = sum((statiche(scena_di[m]) for m in METODI), [])
mobili = list(range(len(fisse), len(fisse) + len(dati(0.0))))
dove = {"FMM": "CPU", "FIM-Jacobi": "GPU", "Block FIM": "GPU"}
fig = make_subplots(rows=1, cols=len(METODI), specs=[[{"type": "scene"}] * len(METODI)], horizontal_spacing=0.01,
                    subplot_titles=[f"{m} ({dove[m]}): arriva a {voli[m]['T'][-1]:.1f} s" for m in METODI])
for tr in fisse + dati(0.0): fig.add_trace(tr)
fig.frames = [go.Frame(data=dati(t), traces=mobili, name=f"{t:.2f}", layout=dict(title=dict(text=titolo(t)))) for t in istanti]
fig.update_layout(title=dict(text=titolo(0.0), x=0.5), **{sc: SCENA for sc in scena_di.values()}, height=1130, margin=dict(l=0, r=0, t=80, b=500),
                  updatemenus=[dict(type="buttons", direction="left", x=0.08, y=0.0, xanchor="right", yanchor="top", pad=dict(r=10, t=60), buttons=[
                      dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=MS_FRAME, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                      dict(label="Pausa", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
                  sliders=[dict(x=0.1, len=0.88, y=0.0, yanchor="top", currentvalue=dict(prefix="tempo di volo t = ", suffix=" s"), pad=dict(t=40),
                                steps=[dict(label=f"{t:.0f}" if abs(t - round(t)) < 1e-9 else "", method="animate",
                                            args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for t in istanti])])
# Riquadro sotto l'animazione: unita' di misura e conversioni, e tabella di velocita' e lentezza nella fascia di sicurezza
M = METRI_PER_UNITA
spessore = (interni[0][1] - interni[0][0]) * M                      # spessore di un muro interno [m]
D_S = D_SICUREZZA * M                                               # fascia di sicurezza [m]
def riga(*celle, larghezze=(17, 11, 9)):                        # una riga della tabella, colonne a larghezza fissa
    return "".join(str(c).rjust(w) for c, w in zip(celle, larghezze))
voci = [("Scala", f"1 unita' = {M:.0f} m; stanza {2 * M:.0f} x {2 * M:.0f} x {2 * M:.0f} m; muri interni spessi {spessore:.0f} m; "
                  f"griglia N = {N}, passo h = {2 / (N - 1) * M * 100:.1f} cm"),
        ("Velocita'", f"v(d) = V_MAX [V_MIN + (1 - V_MIN) min(d / D_S, 1)],  V_MAX = {V_MAX:.0f} m/s, V_MIN = {V_MIN:.1f}, D_S = {D_S:.0f} m"),
        ("", "d = distanza dal muro (il drone e' considerato puntiforme)"),
        ("Lentezza", "F(d) = V_MAX / v(d): F = 1 a velocita' piena"),
        ("Iconale", f"|∇u| = F, u(B) = 0;  tempo di volo da P a B = u(P) x {M:.0f} m / {V_MAX:.0f} m/s = u(P) x {SEC_PER_U:.0f} s")]
testo = "<b>UNITÀ DI MISURA E CONVERSIONI</b><br>" + "<br>".join(f"{k:<11}{v}" for k, v in voci)
testo += f"<br><br><b>Fascia di sicurezza</b> (0 <= d <= D_S = {D_S:.0f} m)<br>"
righe = [riga("d dal muro [m]", "v [m/s]", "F"), riga("-" * 14, "-" * 8, "-" * 6)]
for r in (0.0, 0.25, 0.5, 0.75, 1.0):                               # r = d / D_S, come in lentezza_fm2 (ambiente.h)
    v = V_MAX * (V_MIN + (1 - V_MIN) * r)
    oltre = ">= " if r == 1.0 else ""
    righe.append(riga(f"{oltre}{r * D_S:.2f}", f"{v:.2f}", f"{V_MAX / v:.2f}"))
testo += "<br>".join(righe)
testo += (f"<br><br>Esempio: porte aperte, u(A) = {uA:.3f}, quindi {uA:.3f} x {SEC_PER_U:.0f} s = {uA * SEC_PER_U:.1f} s di volo<br>"
          f"Play: 1 fotogramma = {DT_FRAME} s di volo ogni {MS_FRAME} ms, cioe' {DT_FRAME * 1000 / MS_FRAME:.0f} volte piu' veloce del vero")
fig.add_annotation(xref="paper", yref="paper", x=0.5, y=-0.27, xanchor="center", yanchor="top", align="left", showarrow=False,
                   bgcolor="#f4f6fa", bordercolor="#9a9a9a", borderwidth=1, borderpad=10,
                   font=dict(size=12, family="Courier New, monospace"), text=testo.replace(" ", "\u00a0"))   # spazi fissi: colonne allineate
fig.write_html("ripianificazione_affiancata.html", include_plotlyjs="cdn")    # anche come file da scaricare: si apre nel browser
fig.show()

# 6) VELOCITA' NEL TEMPO e TEMPO A VELOCITA' MASSIMA dei tre droni
g = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32], horizontal_spacing=0.1,
                  subplot_titles=("Velocita' nel tempo (fasce = ricalcola; linee = la mappa cambia)", "Come passa il tempo di volo"))
quote = {}
for met, v in voli.items():
    T, V = v["T"], v["V"]
    g.add_trace(go.Scatter(x=T, y=V, mode="lines", line=dict(color=colore[met], width=2.5), name=f"{met}: arriva a {T[-1]:.1f} s"), row=1, col=1)
    for a, b in v["ricalcolo"]:
        g.add_vrect(x0=a, x1=max(b, a + 0.05), fillcolor=colore[met], opacity=0.2, line_width=0, row=1, col=1)
    dt, vm = np.diff(T), (V[:-1] + V[1:]) / 2
    quote[met] = {"a velocita' massima": dt[vm >= V_PIENA].sum(), "piu' lento": dt[(vm > 0.01) & (vm < V_PIENA)].sum(),
                  "fermo davanti alla porta": dt[vm <= 0.01].sum()}
for met, v in voli.items():
    for i, t_ev in enumerate(v["cambi"]):
        testo = ", ".join(nome for _, nome in cambi_mappa[i]) + ("" if v["ricalcolato"][i] else " (lontano)")
        scritta = dict(annotation_text=testo, annotation_position="top left") if met == "Block FIM" else {}   # una scritta sola per cambio
        g.add_vline(x=t_ev, line=dict(color=colore[met], dash="dot"), row=1, col=1, **scritta)
g.add_hline(y=V_MAX, line=dict(color="gray", dash="dash"), annotation_text="V_MAX", row=1, col=1)
for voce, col in [("a velocita' massima", "#2ca02c"), ("piu' lento", "#bcbd22"), ("fermo davanti alla porta", "#7f7f7f")]:
    g.add_trace(go.Bar(x=list(quote), y=[quote[m][voce] for m in quote], name=voce, marker_color=col,
                       text=[f"{quote[m][voce]:.1f} s" for m in quote], textposition="inside"), row=1, col=2)
g.update_layout(template="plotly_white", barmode="stack", height=430, legend=dict(orientation="h", y=-0.2),
                title="Arrivo in B: " + ", ".join(f"{met} {voli[met]['T'][-1]:.1f} s ({quote[met]['fermo davanti alla porta']:.1f} s fermo)" for met in METODI))
g.update_xaxes(title_text="tempo di volo [s]", row=1, col=1); g.update_yaxes(title_text="velocita' [m/s]", row=1, col=1)
g.update_yaxes(title_text="tempo [s]", row=1, col=2)
g.write_html("ripianificazione_velocita_affiancata.html", include_plotlyjs="cdn")
g.show()
for met, q in quote.items():
    print(f"{met:9s}: " + " | ".join(f"{k} {v:5.1f} s" for k, v in q.items()))